In [1]:
# %pip install pydantic instructor

In [2]:
import sys; print(sys.executable)
import pydantic, instructor
print(pydantic.VERSION)

C:\Users\ASUS\AppData\Local\Programs\Python\Python311\python.exe
2.12.5


In [3]:
from pydantic import BaseModel, field_validator
from datetime import date, time
class ReservationRequest(BaseModel):
    customer_name: str
    party_size: int
    reservation_date: date
    reservation_time: str
    special_request: str | None = None
    
    @field_validator("reservation_time")
    @classmethod
    def check_time_format(cls, v: str) -> str:
        time.fromisoformat(v if len(v) > 5 else v + ":00")  
        return v
    
raw_text = "Здравствуйте! Хотели бы столик на четверых в субботу, 8 ноября, часам к 7 вечера. Меня зовут Нурлан. Если можно, у окна — у нас будет день рождения."

In [4]:
prompt = f'''Извлеки данные из текста заявки на бронирование столика в формате JSON.

Текст: "Добрый вечер, нужен стол на двоих завтра в 20:00, меня зовут Айгерим."
Результат: {{"customer_name": "Айгерим", "party_size": 2, "reservation_date": "2026-10-01", "reservation_time": "20:00:00", "special_request": null}}

Текст: "{raw_text}"
Результат:'''

In [5]:
prompt = f'''Извлеки данные из текста накладной в формате JSON.

Текст: "Накладная № НК-2024/088 от 14.02.2025. Сумма: 32 500 KGS."
Результат: {{"number": "НК-2024/088", "amount": 32500, "currency": "KGS", "issue_date": "2025-02-14"}}

Текст: "{raw_text}"
Результат:'''

print(prompt)

Извлеки данные из текста накладной в формате JSON.

Текст: "Накладная № НК-2024/088 от 14.02.2025. Сумма: 32 500 KGS."
Результат: {"number": "НК-2024/088", "amount": 32500, "currency": "KGS", "issue_date": "2025-02-14"}

Текст: "Здравствуйте! Хотели бы столик на четверых в субботу, 8 ноября, часам к 7 вечера. Меня зовут Нурлан. Если можно, у окна — у нас будет день рождения."
Результат:


In [6]:
import os
from dotenv import load_dotenv
from groq import Groq

load_dotenv(override=True)
groq_client = Groq(api_key=os.environ["GROQ_API_KEY"])

raw_response = groq_client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt}],
)
raw_answer = raw_response.choices[0].message.content
print(raw_answer)

{"name":"Нурлан","people":4,"date":"2026-11-08","time":"19:00","special_request":"у окна","occasion":"день рождения"}


In [7]:
import instructor

client = instructor.from_groq(groq_client)

reservation = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    response_model=ReservationRequest,
    max_retries=2,
    messages=[{"role": "user", "content": raw_text}],
)
print(reservation)

customer_name='Нурлан' party_size=4 reservation_date=datetime.date(2026, 11, 8) reservation_time='19:00' special_request='у окна'
